# 01b RQ1 exploratory: early closes and the 1-day event result

**Exploratory (Section 8).** Nothing here is pre-registered, and none of it changes
the confirmatory RQ1 result in notebook 01.

**Why.** The 1-day event interaction (+0.39) falls to +0.13, with a CI covering 0,
when markets with `closedTime < end_date` are dropped (sensitivity 5.7). Two-thirds of
the crypto markets in that cell closed early, against about a quarter of the
non-crypto ones.

**Candidate mechanism (stated before the breakdowns below were run).** In a
deadline market ("will X hit Y by date"), the market closes as soon as the event
occurs. So an early close mostly means the event happened, and `closedTime` is set
by the outcome itself. A snapshot taken one day before such a close lands one day
before the event. That point is chosen using the future, so even well-calibrated
prices would look too timid there (slope > 1, outcomes more frequent than priced).
If this is what is happening, then among early-closed markets:

1. the listed outcome (token1, usually "Yes") wins far more often than on time, and
   more so for crypto;
2. calibration-in-the-large is positive (outcomes more frequent than priced);
3. the crypto × non-crypto slope gap sits in the early closes, not in the on-time
   markets.

Sports early closes, by contrast, usually reflect a placeholder `end_date`, not an
outcome-triggered close. That asymmetry would explain why the effect shows up as a
crypto difference.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.special import logit
from statsmodels.tools.sm_exceptions import ConvergenceWarning, PerfectSeparationWarning

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
CLIP = (0.001, 0.999)
SEED = 20261112
HORIZONS = ["1w", "1d", "1h"]
FORMATS = ["template", "event"]
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

frame = pd.read_parquet(DATA / "mart_analysis.parquet")
frame = frame[frame["is_primary"]].copy()
markets = pd.read_parquet(DATA / "stg_markets.parquet", columns=["market_id", "question", "answer1"])
frame = frame.merge(markets, on="market_id", how="left", validate="many_to_one")
frame["y"] = frame["token1_won"].astype(float)
frame["x"] = logit(frame["snap_price"].clip(*CLIP))
frame["lead_days"] = (frame["end_date"] - frame["closed_time"]).dt.total_seconds() / 86400
cell = frame[(frame["horizon"] == "1d") & (frame["format"] == "event")]
len(cell)

197712

## E1. Who closes early, and how do they resolve?

Per crypto × early close in the 1-day event cell: markets, clusters, median lead time
(days between `closedTime` and `end_date`), the share whose first-listed answer is
"Yes", the share where token1 won, and the mean snapshot price. Then the same split by
subject.

In [2]:
def profile(d, by):
    return d.groupby(by).agg(
        markets=("y", "size"), clusters=("cluster_id", "nunique"),
        median_lead_days=("lead_days", "median"),
        answer1_yes=("answer1", lambda s: (s.str.lower() == "yes").mean()),
        token1_won=("y", "mean"), mean_price=("snap_price", "mean"),
    )


e1 = profile(cell, ["is_crypto", "early_close"])
e1_subject = profile(cell, ["subject", "early_close"])
e1.to_csv(RESULTS / "rq1_early_close_profile.csv")
display(e1.round(3), e1_subject.round(3))

markets  clusters  median_lead_days  answer1_yes  token1_won  mean_price
is_crypto early_close                                                                          
False     False         141082     10312            -0.204        0.575       0.365       0.370
          True           54188      6818             6.666        0.421       0.433       0.426
True      False            840       456            -0.794        0.995       0.095       0.114
          True            1602       416           138.741        0.975       0.443       0.404

markets  clusters  median_lead_days  answer1_yes  token1_won  mean_price
subject  early_close                                                                          
crypto   False            840       456            -0.794        0.995       0.095       0.114
         True            1602       416           138.741        0.975       0.443       0.404
other    False          17348      4917            -0.850        0.995       0.272       0.283
         True            9807      2281            11.761        0.997       0.387       0.350
politics False          11516      3004            -1.071        0.993       0.300       0.325
         True            4209      1036            16.699        0.993       0.426       0.359
sports   False         112094      2551            -0.186        0.467       0.386       0.389
         True           40128      3557             6.609        0.220       0.445       0.451
weather  False            124        27            -1.658        1.000       0.250       0.276
         True              44        28            15.964        1.000       0.659       0.604

## E2. The registered model, split by early vs. on-time close

Same specification as notebook 01 (`token1_won ~ logit(p) + crypto + crypto × logit(p)`,
CR1 cluster SEs), fitted separately to early-closed and on-time markets in every
(horizon × format) cell. Coefficients and 95% CIs only. A cell with fewer than 30
clusters on either side is not fitted, and a fit that does not converge is blanked.

In [3]:
def fit(d, cols):
    """Logit with CR1 cluster SEs; returns params, cov, ok (converged, no separation)."""
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        res = sm.Logit(d["y"].to_numpy(), np.column_stack(cols)).fit(
            disp=0, maxiter=200, cov_type="cluster",
            cov_kwds={"groups": pd.factorize(d["cluster_id"])[0]})
    bad = any(issubclass(w.category, (ConvergenceWarning, PerfectSeparationWarning)) for w in caught)
    return res.params, res.cov_params(), bool(res.mle_retvals["converged"]) and not bad


def split_fit(d):
    c = d["is_crypto"]
    out = {"n_crypto": int(c.sum()), "n_noncrypto": int((~c).sum()),
           "clusters_crypto": d.loc[c, "cluster_id"].nunique(),
           "clusters_noncrypto": d.loc[~c, "cluster_id"].nunique()}
    if min(out["clusters_crypto"], out["clusters_noncrypto"]) < 30:
        return {**out, "fitted": False}
    x, cf = d["x"].to_numpy(), c.to_numpy().astype(float)
    b, V, ok = fit(d, [np.ones_like(x), x, cf, cf * x])
    if not ok:
        return {**out, "fitted": False}
    se = np.sqrt(V[3, 3])
    return {**out, "fitted": True, "slope_noncrypto": b[1], "slope_crypto": b[1] + b[3],
            "interaction": b[3], "ci_low": b[3] - 1.959964 * se, "ci_high": b[3] + 1.959964 * se}


rows = []
for h in HORIZONS:
    for f in FORMATS:
        for early in (True, False):
            d = frame[(frame["horizon"] == h) & (frame["format"] == f) & (frame["early_close"] == early)]
            rows.append({"horizon": h, "format": f, "early_close": early, **split_fit(d)})
e2 = pd.DataFrame(rows)
e2.to_csv(RESULTS / "rq1_early_close_split.csv", index=False)
e2.round(3)

,horizon,format,early_close,n_crypto,n_noncrypto,clusters_crypto,clusters_noncrypto,fitted,slope_noncrypto,slope_crypto,interaction,ci_low,ci_high
0,1w,template,True,354,2545,20,77,False,NaN,NaN,NaN,NaN,NaN
1,1w,template,False,5448,7350,159,257,True,0.882,0.988,0.106,-0.093,0.304
2,1w,event,True,1133,18589,339,3792,True,1.020,1.065,0.045,-0.108,0.198
3,1w,event,False,915,41822,466,7760,True,0.943,1.159,0.216,-0.078,0.511
4,1d,template,True,1140,16069,45,197,True,1.512,1.007,-0.505,-0.680,-0.330
5,1d,template,False,35982,100632,244,395,True,1.036,1.066,0.030,-0.013,0.073
6,1d,event,True,1602,54188,416,6818,True,0.991,1.446,0.456,0.217,0.695
7,1d,event,False,840,141082,456,10312,True,0.971,1.096,0.125,-0.069,0.320
8,1h,template,True,1303,7492,55,156,True,1.626,0.718,-0.908,-1.181,-0.635
9,1h,template,False,102986,27690,190,276,True,1.241,1.121,-0.119,-0.212,-0.027


## E3. Is the crypto gap different in early closes? (1-day event)

A single model in the 1-day event cell adds the early-close indicator e and all its
interactions: `y ~ x · crypto · e`. The coefficient on crypto × e × logit(p) measures
how much bigger the crypto slope gap is in early-closed markets than in on-time
ones. 95% CI only.

In [4]:
x = cell["x"].to_numpy()
c = cell["is_crypto"].to_numpy().astype(float)
e = cell["early_close"].to_numpy().astype(float)
names = ["const", "x", "crypto", "crypto:x", "early", "early:x", "crypto:early", "crypto:early:x"]
b, V, ok = fit(cell, [np.ones_like(x), x, c, c * x, e, e * x, c * e, c * e * x])
se = np.sqrt(np.diag(V))
e3 = pd.DataFrame({"term": names, "coef": b, "ci_low": b - 1.959964 * se, "ci_high": b + 1.959964 * se})
print("converged without separation:", ok)
e3.to_csv(RESULTS / "rq1_early_close_triple.csv", index=False)
e3.round(3)

converged without separation: True


,term,coef,ci_low,ci_high
0,const,-0.044,-0.065,-0.022
1,x,0.971,0.953,0.989
2,crypto,-0.742,-1.236,-0.249
3,crypto:x,0.125,-0.069,0.320
4,early,0.089,0.049,0.129
5,early:x,0.020,-0.026,0.065
6,crypto:early,1.547,0.969,2.126
7,crypto:early:x,0.330,0.020,0.641


## E4. Calibration-in-the-large by group (1-day event)

The intercept a in logit P(win) = a + logit(p), with logit(p) as an offset (D7.2), fitted
per crypto × early close with CR1 cluster SEs. a > 0 means outcomes happen more often
than priced.

In [5]:
rows = []
for (crypto, early), d in cell.groupby(["is_crypto", "early_close"]):
    res = sm.GLM(d["y"].to_numpy(), np.ones(len(d)), family=sm.families.Binomial(),
                 offset=d["x"].to_numpy()).fit(cov_type="cluster",
                                               cov_kwds={"groups": pd.factorize(d["cluster_id"])[0]})
    a, s = res.params[0], res.bse[0]
    rows.append({"is_crypto": crypto, "early_close": early, "markets": len(d),
                 "citl": a, "ci_low": a - 1.959964 * s, "ci_high": a + 1.959964 * s})
e4 = pd.DataFrame(rows)
e4.to_csv(RESULTS / "rq1_early_close_citl.csv", index=False)
e4.round(3)

,is_crypto,early_close,markets,citl,ci_low,ci_high
0,False,False,141082,-0.034,-0.053,-0.016
1,False,True,54188,0.046,0.011,0.081
2,True,False,840,-0.801,-1.235,-0.368
3,True,True,1602,0.578,0.366,0.791


## E5. What the early-closed crypto markets are

Twelve randomly drawn (fixed seed) crypto early closes from the 1-day event cell, for a
plain read of the question wording, plus the lead-time distribution.

In [6]:
ce = cell[cell["is_crypto"] & cell["early_close"]]
print(ce["lead_days"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(2))
ce.sample(12, random_state=SEED)[["question", "answer1", "snap_price", "token1_won", "lead_days"]].round(2)

count    1602.00
mean      209.23
std       198.38
min         0.00
10%         6.94
25%        27.55
50%       138.74
75%       343.24
90%       548.31
max       686.20
Name: lead_days, dtype: float64


,question,answer1,snap_price,token1_won,lead_days
345110,"Will BTC hit $55,000 in February?",Yes,0.18,True,1.83
402986,Over $30M committed to the Credible public sale?,Yes,0.91,True,45.25
636726,Will Coinbase announce Popcat as next listing?,Yes,0.06,False,14.64
663142,Will Ethereum reach $2200 in May?,Yes,0.21,True,22.43
129664,Perle Labs FDV above $300M one day after launch?,Yes,0.01,False,645.29
614811,Ethena token FDV between 5b and 7.5b one day after launch?,Yes,0.10,False,272.07
437047,RipCars FDV above $2M one day after launch?,Yes,0.49,False,158.28
23675,Over $30M committed to the Ranger public sale?,Yes,0.92,True,21.46
44188,Elsa FDV above $200M one day after launch?,Yes,0.02,False,343.85
717187,Will Monad perform an airdrop by November 15?,Yes,0.00,False,150.72


## E6. Which crypto questions carry it? (crude keyword split)

E5 shows two kinds of crypto early close: "FDV above X one day after launch" markets,
which close when the launch happens whatever the outcome, and deadline markets ("hit
$X by date"), which close when the event occurs. The split below is a **crude keyword
rule, not the validated classifier**:

- `launch_fdv` matches fdv / market cap / after launch;
- `deadline_hit` matches by / before / reach / hit / all time high;
- `launch_other` matches airdrop / launch / list;
- everything else is `other`.

The registered 1-day event interaction is then refitted with each crypto kind
removed in turn.

In [7]:
q = cell["question"].str.lower()
cell = cell.assign(kind=np.select(
    [q.str.contains("fdv|market cap|after launch"),
     q.str.contains(r"\bby\b|before|reach|hit|all time high|\bath\b"),
     q.str.contains("airdrop|launch|list")],
    ["launch_fdv", "deadline_hit", "launch_other"], "other"))
e6_mix = (cell[cell["is_crypto"]].groupby(["early_close", "kind"])
          .agg(markets=("y", "size"), token1_won=("y", "mean"), mean_price=("snap_price", "mean")))
rows = [{"dropped": "none", **split_fit(cell)}]
for k in ["launch_fdv", "deadline_hit", "launch_other", "other"]:
    rows.append({"dropped": f"crypto {k}", **split_fit(cell[~(cell["is_crypto"] & (cell["kind"] == k))])})
e6 = pd.DataFrame(rows)
e6_mix.to_csv(RESULTS / "rq1_early_close_kinds.csv")
e6.to_csv(RESULTS / "rq1_early_close_drop_kind.csv", index=False)
display(e6_mix.round(3), e6.round(3))

markets  token1_won  mean_price
early_close kind                                         
False       deadline_hit      480       0.033       0.046
            launch_fdv         86       0.186       0.202
            launch_other       80       0.050       0.069
            other             194       0.227       0.261
True        deadline_hit      338       0.793       0.638
            launch_fdv        640       0.328       0.331
            launch_other      102       0.284       0.255
            other             522       0.389       0.370

,dropped,n_crypto,n_noncrypto,clusters_crypto,clusters_noncrypto,fitted,slope_noncrypto,slope_crypto,interaction,ci_low,ci_high
0,none,2442,195270,816,15852,True,0.98,1.369,0.388,0.210,0.567
1,crypto launch_fdv,1716,195270,695,15852,True,0.98,1.298,0.318,0.131,0.505
2,crypto deadline_hit,1624,195270,395,15852,True,0.98,1.331,0.350,0.135,0.566
3,crypto launch_other,2260,195270,762,15852,True,0.98,1.384,0.404,0.215,0.593
4,crypto other,1726,195270,617,15852,True,0.98,1.498,0.518,0.268,0.767


## Summary (exploratory)

- **The prediction holds where it can be checked.** Crypto early closes have positive
  calibration-in-the-large and on-time crypto markets have negative CITL (E4). The gap
  is largest in deadline-style questions (E6). The crypto slope gap is larger in early
  closes than on time (E3, crypto × early × logit(p) CI just above 0).
- **But the split is itself selected on the outcome.** In a deadline market, closing
  early mostly means "Yes" and running to the deadline mostly means "No". Conditioning
  on `early_close` therefore biases *both* subsets, in opposite directions. Sensitivity
  5.7 (on-time markets only) is not a clean correction; it keeps the mirror-image half.
  E2 to E4 are consistent with outcome-triggered anchoring, but they cannot isolate it.
- **No single question type carries the result.** Dropping the deadline-style or the
  launch/FDV crypto markets in turn leaves the 1-day event interaction positive, with
  a CI above 0 (E6).
- **Reading for the write-up:** the confirmatory 1-day event result stands as
  registered. It is fragile to sensitivity 5.7, and part of it plausibly reflects
  horizon anchoring at an outcome-triggered `closedTime` rather than mispricing a
  trader could exploit. A clean check would need an anchor that does not depend on
  the outcome. For example, a fixed calendar time before `end_date` only for markets
  whose `end_date` is a true deadline. That is a new design, not a sensitivity, and
  is not attempted here.